In [ ]:
import json
import numpy as np
import torch
import warnings
from typing import List, Dict, Tuple
from collections import defaultdict
from pathlib import Path

warnings.filterwarnings('ignore')

# Install required packages
import subprocess
import sys

def install_packages():
    packages = ['transformers', 'torch', 'datasets', 'scikit-learn', 'seqeval']
    for package in packages:
        try:
            __import__(package)
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

install_packages()

from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

print("All packages installed successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:

# STEP 1: Load and explore training dataset

# Load training data
with open('2.2_train_dataset.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

print("="*70)
print("DATASET EXPLORATION")
print("="*70)
print(f"\nTotal training samples: {len(train_data)}")

# Show statistics
roles_count = defaultdict(int)
for sample in train_data:
    for role, text in sample['roles'].items():
        if text.strip():
            roles_count[role] += 1

print("\nRole coverage:")
for role in ['Agent', 'Theme', 'Recipient', 'Time', 'Condition']:
    filled = roles_count[role]
    pct = 100 * filled / len(train_data)
    print(f"  {role:12s}: {filled:3d}/{len(train_data)} ({pct:5.1f}%)")

# Display sample
print("\nSample clause:")
sample = train_data[0]
print(f"  Text: {sample['text']}")
print(f"  Predicate: {sample['predicate']}")
print(f"  Roles:")
for role, text in sample['roles'].items():
    if text.strip():
        print(f"    - {role}: {text}")

# Split into train/test
from sklearn.model_selection import train_test_split
train_samples, test_samples = train_test_split(train_data, test_size=0.2, random_state=42)
print(f"\nTrain/Test split: {len(train_samples)}/{len(test_samples)}")


In [ ]:

# STEP 2: Convert to BIO tagging scheme

class BIOTagger:
    """Convert role text spans to BIO tags"""

    ROLES = ['Agent', 'Theme', 'Recipient', 'Time', 'Condition']
    LABEL2ID = {
        'O': 0,
        'B-Agent': 1, 'I-Agent': 2,
        'B-Theme': 3, 'I-Theme': 4,
        'B-Recipient': 5, 'I-Recipient': 6,
        'B-Time': 7, 'I-Time': 8,
        'B-Condition': 9, 'I-Condition': 10,
    }
    ID2LABEL = {v: k for k, v in LABEL2ID.items()}

    @staticmethod
    def align_tokens_to_roles(text: str, roles: Dict[str, str], tokens: List[str]) -> List[int]:
        """Align BIO tags to tokens without offset_mapping"""
        labels = [0] * len(tokens)  # Initialize all as 'O'

        # Skip special tokens
        special_tokens = {'[CLS]', '[SEP]', '[PAD]', '<s>', '</s>', '<unk>'}

        # Find role positions
        role_spans = {}
        for role, role_text in roles.items():
            if not role_text.strip():
                continue
            start = text.lower().find(role_text.lower())
            if start >= 0:
                end = start + len(role_text)
                role_spans[role] = (start, end)

        if not role_spans:
            return labels

        # Map tokens to character positions
        token_positions = []
        char_idx = 0

        for token in tokens:
            if token in special_tokens:
                token_positions.append(None)
                continue

            clean_token = token.replace('▁', ' ').strip()
            if not clean_token:
                token_positions.append(None)
                continue

            # Find token position
            start = text.lower().find(clean_token.lower(), char_idx)
            if start >= 0:
                end = start + len(clean_token)
                token_positions.append((start, end))
                char_idx = end
            else:
                token_positions.append(None)

        # Assign labels
        for role, (role_start, role_end) in role_spans.items():
            is_first = True
            for i, token_pos in enumerate(token_positions):
                if token_pos is None:
                    continue

                token_start, token_end = token_pos

                # Check overlap
                if token_start < role_end and token_end > role_start:
                    if is_first:
                        labels[i] = BIOTagger.LABEL2ID[f'B-{role}']
                        is_first = False
                    else:
                        labels[i] = BIOTagger.LABEL2ID[f'I-{role}']

        return labels

    @staticmethod
    def convert_dataset(samples: List[Dict], tokenizer) -> List[Dict]:
        """Convert all samples to token-labeled format"""
        dataset = []

        for sample in samples:
            text = sample['text']
            roles = sample['roles']

            # Tokenize
            encoding = tokenizer(
                text,
                max_length=512,
                padding='max_length',
                truncation=True
            )

            # Get tokens
            tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])

            # Align roles to tokens
            labels = BIOTagger.align_tokens_to_roles(text, roles, tokens)

            # Pad/truncate to 512
            labels = (labels + [0] * 512)[:512]

            dataset.append({
                'input_ids': encoding['input_ids'],
                'attention_mask': encoding['attention_mask'],
                'token_type_ids': encoding.get('token_type_ids', [0] * 512),
                'labels': labels
            })

        return dataset

# Initialize tokenizer
model_name = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Tokenizer loaded: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Convert datasets
print("\nConverting datasets to BIO format...")
try:
    train_dataset = BIOTagger.convert_dataset(train_samples, tokenizer)
    test_dataset = BIOTagger.convert_dataset(test_samples, tokenizer)

    print(f"Training samples prepared: {len(train_dataset)}")
    print(f"Test samples prepared: {len(test_dataset)}")
    print(f"\nLabel mapping:")
    for label, id in list(BIOTagger.LABEL2ID.items())[:6]:
        print(f"  {label}: {id}")
    print(f"  ...")
    print(f"\nDataset ready for training!")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()


In [ ]:

# STEP 3: Prepare model and training

# Convert to HuggingFace Dataset
def dict_to_dataset(data_list):
    """Convert list of dicts to HuggingFace Dataset"""
    return Dataset.from_dict({
        'input_ids': [d['input_ids'] for d in data_list],
        'attention_mask': [d['attention_mask'] for d in data_list],
        'token_type_ids': [d['token_type_ids'] for d in data_list],
        'labels': [d['labels'] for d in data_list]
    })

train_hf_dataset = dict_to_dataset(train_dataset)
test_hf_dataset = dict_to_dataset(test_dataset)

print(f"HuggingFace datasets created")
print(f"Train dataset features: {train_hf_dataset.features}")

# Load model
num_labels = len(BIOTagger.LABEL2ID)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

print(f"\nModel loaded: {model_name}")
print(f"Number of labels: {num_labels}")
print(f"Model parameters: {model.num_parameters():,}")

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model moved to: {device}")


In [ ]:

# STEP 4: Train model

# Define evaluation metrics
def compute_metrics(p):
    """Compute evaluation metrics"""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (special tokens)
    true_predictions = [
        [BIOTagger.ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [BIOTagger.ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions)
    }

# Training arguments
training_args = TrainingArguments(
    output_dir='./phobert_srl_model',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=10
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf_dataset,
    eval_dataset=test_hf_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training...")
print("="*70)
history = trainer.train()
print("="*70)
print("Training completed!")

# Save model
model.save_pretrained('./phobert_srl_model/final')
tokenizer.save_pretrained('./phobert_srl_model/final')
print("\nModel saved to: ./phobert_srl_model/final")

# STEP 5: Evaluate on test set

print("="*70)
print("EVALUATING MODEL")
print("="*70)

# Evaluate
try:
    results = trainer.evaluate()
    
    print("\nTEST SET RESULTS")
    print("="*70)
    for key, value in results.items():
        if 'loss' in key:
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value:.4f}")
    
except RuntimeError as e:
    print(f"Note: Training not yet complete. Skipping trainer evaluation.")
    print("Run this cell AFTER training completes.")
    results = {}

# Manual evaluation
print("\nManual Prediction Evaluation:")
print("="*70)

predictions = trainer.predict(test_hf_dataset)
preds = np.argmax(predictions.predictions, axis=2)

true_predictions = []
true_labels_list = []

for pred, label in zip(preds, predictions.label_ids):
    true_pred = [BIOTagger.ID2LABEL[p] for (p, l) in zip(pred, label) if l != -100]
    true_label = [BIOTagger.ID2LABEL[l] for (p, l) in zip(pred, label) if l != -100]
    true_predictions.append(true_pred)
    true_labels_list.append(true_label)

print("\nDETAILED CLASSIFICATION REPORT")
print("="*70)
report = classification_report(true_labels_list, true_predictions)
print(report)

# Save results
with open('phobert_srl_results.txt', 'w', encoding='utf-8') as f:
    f.write("TEST SET RESULTS\n")
    f.write("="*70 + "\n\n")
    if results:
        for key, value in results.items():
            if 'loss' in key:
                f.write(f"{key}: {value:.4f}\n")
            else:
                f.write(f"{key}: {value:.4f}\n")
    f.write("\n" + "="*70 + "\n")
    f.write("DETAILED CLASSIFICATION REPORT\n")
    f.write("="*70 + "\n\n")
    f.write(report)

print("\nResults saved to: phobert_srl_results.txt")


In [ ]:

# STEP 6: Prediction function and test

class SRLPredictor:
    """Predict semantic roles for new text"""
    
    def __init__(self, model_path='./phobert_srl_model/final'):
        self.model = AutoModelForTokenClassification.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        self.model.eval()
    
    def predict(self, text: str) -> Dict[str, str]:
        """Predict roles for a clause"""
        encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True
        )
        
        input_ids = torch.tensor([encoding['input_ids']]).to(self.device)
        attention_mask = torch.tensor([encoding['attention_mask']]).to(self.device)
        
        # Predict
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        
        logits = outputs.logits[0]
        predictions = torch.argmax(logits, dim=-1).cpu().numpy()
        
        # Convert predictions to roles
        roles = {role: [] for role in BIOTagger.ROLES}
        current_role = None
        current_text = []
        
        tokens = self.tokenizer.convert_ids_to_tokens(encoding['input_ids'])
        
        for token_idx, (pred, token) in enumerate(zip(predictions, tokens)):
            label = BIOTagger.ID2LABEL[pred]
            
            # Skip special tokens
            if token in {'[CLS]', '[SEP]', '[PAD]', '<s>', '</s>'}:
                continue
            
            # Clean token
            clean_token = token.replace('▁', ' ').strip()
            if not clean_token:
                continue
            
            if label == 'O':
                # Outside any role - save current role if exists
                if current_role and current_text:
                    roles[current_role].append(' '.join(current_text))
                    current_role = None
                    current_text = []
            else:
                # Inside a role (B-X or I-X)
                role = label.split('-')[1]
                
                if label.startswith('B-'):
                    # Beginning of role - save previous role if different
                    if current_role and current_role != role and current_text:
                        roles[current_role].append(' '.join(current_text))
                        current_text = []
                    current_role = role
                    current_text = [clean_token]
                else:  # I-X
                    # Continuation of role
                    if current_role == role:
                        current_text.append(clean_token)
                    else:
                        if current_role and current_text:
                            roles[current_role].append(' '.join(current_text))
                        current_role = role
                        current_text = [clean_token]
        
        # Save last role
        if current_role and current_text:
            roles[current_role].append(' '.join(current_text))
        
        # Consolidate and clean
        result = {}
        for role, texts in roles.items():
            if texts:
                result[role] = ', '.join(texts)
            else:
                result[role] = ''
        
        return result

# Test prediction
print("="*70)
print("TESTING PREDICTION")
print("="*70)

predictor = SRLPredictor()

# Test on a few samples
test_texts = [
    test_samples[0]['text'],
    test_samples[1]['text'] if len(test_samples) > 1 else test_samples[0]['text'],
]

for i, text in enumerate(test_texts, 1):
    print(f"\nSample {i}:")
    print(f"Text: {text[:70]}...")
    
    # Ground truth
    sample = test_samples[i-1]
    print(f"Ground truth roles:")
    has_gt = False
    for role, role_text in sample['roles'].items():
        if role_text.strip():
            print(f"  {role}: {role_text[:50]}...")
            has_gt = True
    if not has_gt:
        print("  (No roles)")
    
    # Predictions
    predicted_roles = predictor.predict(text)
    print(f"Predicted roles:")
    has_pred = False
    for role, role_text in predicted_roles.items():
        if role_text.strip():
            print(f"  {role}: {role_text[:50]}...")
            has_pred = True
    if not has_pred:
        print("  (No roles)")

print("\n" + "="*70)
print("Model ready for inference!")
print("="*70)

In [ ]:
# STEP 2: Convert to BIO tagging scheme

class BIOTagger:
    """Convert role text spans to BIO tags"""

    ROLES = ['Agent', 'Theme', 'Recipient', 'Time', 'Condition']
    LABEL2ID = {
        'O': 0,
        'B-Agent': 1, 'I-Agent': 2,
        'B-Theme': 3, 'I-Theme': 4,
        'B-Recipient': 5, 'I-Recipient': 6,
        'B-Time': 7, 'I-Time': 8,
        'B-Condition': 9, 'I-Condition': 10,
    }
    ID2LABEL = {v: k for k, v in LABEL2ID.items()}

    @staticmethod
    def align_tokens_to_roles(text: str, roles: Dict[str, str], tokens: List[str]) -> List[int]:
        """Align BIO tags to tokens without offset_mapping"""
        labels = [0] * len(tokens)  # Initialize all as 'O'

        # Skip special tokens
        special_tokens = {'[CLS]', '[SEP]', '[PAD]', '<s>', '</s>', '<unk>'}

        # Find role positions
        role_spans = {}
        for role, role_text in roles.items():
            if not role_text.strip():
                continue
            start = text.lower().find(role_text.lower())
            if start >= 0:
                end = start + len(role_text)
                role_spans[role] = (start, end)

        if not role_spans:
            return labels

        # Map tokens to character positions
        token_positions = []
        char_idx = 0

        for token in tokens:
            if token in special_tokens:
                token_positions.append(None)
                continue

            clean_token = token.replace('▁', ' ').strip()
            if not clean_token:
                token_positions.append(None)
                continue

            # Find token position
            start = text.lower().find(clean_token.lower(), char_idx)
            if start >= 0:
                end = start + len(clean_token)
                token_positions.append((start, end))
                char_idx = end
            else:
                token_positions.append(None)

        # Assign labels
        for role, (role_start, role_end) in role_spans.items():
            is_first = True
            for i, token_pos in enumerate(token_positions):
                if token_pos is None:
                    continue

                token_start, token_end = token_pos

                # Check overlap
                if token_start < role_end and token_end > role_start:
                    if is_first:
                        labels[i] = BIOTagger.LABEL2ID[f'B-{role}']
                        is_first = False
                    else:
                        labels[i] = BIOTagger.LABEL2ID[f'I-{role}']

        return labels

    @staticmethod
    def convert_dataset(samples: List[Dict], tokenizer) -> List[Dict]:
        """Convert all samples to token-labeled format"""
        dataset = []

        for sample in samples:
            text = sample['text']
            roles = sample['roles']

            # Tokenize
            encoding = tokenizer(
                text,
                max_length=512,
                padding='max_length',
                truncation=True
            )

            # Get tokens
            tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])

            # Align roles to tokens
            labels = BIOTagger.align_tokens_to_roles(text, roles, tokens)

            # Pad/truncate to 512
            labels = (labels + [0] * 512)[:512]

            dataset.append({
                'input_ids': encoding['input_ids'],
                'attention_mask': encoding['attention_mask'],
                'token_type_ids': encoding.get('token_type_ids', [0] * 512),
                'labels': labels
            })

        return dataset

# Initialize tokenizer
model_name = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Tokenizer loaded: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Convert datasets
print("\nConverting datasets to BIO format...")
try:
    train_dataset = BIOTagger.convert_dataset(train_samples, tokenizer)
    test_dataset = BIOTagger.convert_dataset(test_samples, tokenizer)

    print(f"Training samples prepared: {len(train_dataset)}")
    print(f"Test samples prepared: {len(test_dataset)}")
    print(f"\nLabel mapping:")
    for label, id in list(BIOTagger.LABEL2ID.items())[:6]:
        print(f"  {label}: {id}")
    print(f"  ...")
    print(f"\nDataset ready for training!")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

# STEP 3: Prepare model and training

# Convert to HuggingFace Dataset
def dict_to_dataset(data_list):
    """Convert list of dicts to HuggingFace Dataset"""
    return Dataset.from_dict({
        'input_ids': [d['input_ids'] for d in data_list],
        'attention_mask': [d['attention_mask'] for d in data_list],
        'token_type_ids': [d['token_type_ids'] for d in data_list],
        'labels': [d['labels'] for d in data_list]
    })

train_hf_dataset = dict_to_dataset(train_dataset)
test_hf_dataset = dict_to_dataset(test_dataset)

print(f"HuggingFace datasets created")
print(f"Train dataset features: {train_hf_dataset.features}")

# Load model
num_labels = len(BIOTagger.LABEL2ID)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

print(f"\nModel loaded: {model_name}")
print(f"Number of labels: {num_labels}")
print(f"Model parameters: {model.num_parameters():,}")

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model moved to: {device}")

# STEP 4: Train model

# Define evaluation metrics
def compute_metrics(p):
    """Compute evaluation metrics"""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (special tokens)
    true_predictions = [
        [BIOTagger.ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [BIOTagger.ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions)
    }

# Training arguments
training_args = TrainingArguments(
    output_dir='./phobert_srl_model',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=10
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf_dataset,
    eval_dataset=test_hf_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training...")
print("="*70)
history = trainer.train()
print("="*70)
print("Training completed!")

# Save model
model.save_pretrained('./phobert_srl_model/final')
tokenizer.save_pretrained('./phobert_srl_model/final')
print("\nModel saved to: ./phobert_srl_model/final")

# STEP 5: Evaluate on test set

print("="*70)
print("EVALUATING MODEL")
print("="*70)

# Evaluate
try:
    results = trainer.evaluate()
    
    print("\nTEST SET RESULTS")
    print("="*70)
    for key, value in results.items():
        if 'loss' in key:
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value:.4f}")
    
except RuntimeError as e:
    print(f"Note: Training not yet complete. Skipping trainer evaluation.")
    print("Run this cell AFTER training completes.")
    results = {}

# Manual evaluation
print("\nManual Prediction Evaluation:")
print("="*70)

predictions = trainer.predict(test_hf_dataset)
preds = np.argmax(predictions.predictions, axis=2)

true_predictions = []
true_labels_list = []

for pred, label in zip(preds, predictions.label_ids):
    true_pred = [BIOTagger.ID2LABEL[p] for (p, l) in zip(pred, label) if l != -100]
    true_label = [BIOTagger.ID2LABEL[l] for (p, l) in zip(pred, label) if l != -100]
    true_predictions.append(true_pred)
    true_labels_list.append(true_label)

print("\nDETAILED CLASSIFICATION REPORT")
print("="*70)
report = classification_report(true_labels_list, true_predictions)
print(report)

# Save results
with open('phobert_srl_results.txt', 'w', encoding='utf-8') as f:
    f.write("TEST SET RESULTS\n")
    f.write("="*70 + "\n\n")
    if results:
        for key, value in results.items():
            if 'loss' in key:
                f.write(f"{key}: {value:.4f}\n")
            else:
                f.write(f"{key}: {value:.4f}\n")
    f.write("\n" + "="*70 + "\n")
    f.write("DETAILED CLASSIFICATION REPORT\n")
    f.write("="*70 + "\n\n")
    f.write(report)

print("\nResults saved to: phobert_srl_results.txt")

# STEP 6: Prediction function and test

class SRLPredictor:
    """Predict semantic roles for new text"""
    
    def __init__(self, model_path='./phobert_srl_model/final'):
        self.model = AutoModelForTokenClassification.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        self.model.eval()
    
    def predict(self, text: str) -> Dict[str, str]:
        """Predict roles for a clause"""
        encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True
        )
        
        input_ids = torch.tensor([encoding['input_ids']]).to(self.device)
        attention_mask = torch.tensor([encoding['attention_mask']]).to(self.device)
        
        # Predict
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        
        logits = outputs.logits[0]
        predictions = torch.argmax(logits, dim=-1).cpu().numpy()
        
        # Convert predictions to roles
        roles = {role: [] for role in BIOTagger.ROLES}
        current_role = None
        current_text = []
        
        tokens = self.tokenizer.convert_ids_to_tokens(encoding['input_ids'])
        
        for token_idx, (pred, token) in enumerate(zip(predictions, tokens)):
            label = BIOTagger.ID2LABEL[pred]
            
            # Skip special tokens
            if token in {'[CLS]', '[SEP]', '[PAD]', '<s>', '</s>'}:
                continue
            
            # Clean token
            clean_token = token.replace('▁', ' ').strip()
            if not clean_token:
                continue
            
            if label == 'O':
                # Outside any role - save current role if exists
                if current_role and current_text:
                    roles[current_role].append(' '.join(current_text))
                    current_role = None
                    current_text = []
            else:
                # Inside a role (B-X or I-X)
                role = label.split('-')[1]
                
                if label.startswith('B-'):
                    # Beginning of role - save previous role if different
                    if current_role and current_role != role and current_text:
                        roles[current_role].append(' '.join(current_text))
                        current_text = []
                    current_role = role
                    current_text = [clean_token]
                else:  # I-X
                    # Continuation of role
                    if current_role == role:
                        current_text.append(clean_token)
                    else:
                        if current_role and current_text:
                            roles[current_role].append(' '.join(current_text))
                        current_role = role
                        current_text = [clean_token]
        
        # Save last role
        if current_role and current_text:
            roles[current_role].append(' '.join(current_text))
        
        # Consolidate and clean
        result = {}
        for role, texts in roles.items():
            if texts:
                result[role] = ', '.join(texts)
            else:
                result[role] = ''
        
        return result

# Test prediction
print("="*70)
print("TESTING PREDICTION")
print("="*70)

predictor = SRLPredictor()

# Test on a few samples
test_texts = [
    test_samples[0]['text'],
    test_samples[1]['text'] if len(test_samples) > 1 else test_samples[0]['text'],
]

for i, text in enumerate(test_texts, 1):
    print(f"\nSample {i}:")
    print(f"Text: {text[:70]}...")
    
    # Ground truth
    sample = test_samples[i-1]
    print(f"Ground truth roles:")
    has_gt = False
    for role, role_text in sample['roles'].items():
        if role_text.strip():
            print(f"  {role}: {role_text[:50]}...")
            has_gt = True
    if not has_gt:
        print("  (No roles)")
    
    # Predictions
    predicted_roles = predictor.predict(text)
    print(f"Predicted roles:")
    has_pred = False
    for role, role_text in predicted_roles.items():
        if role_text.strip():
            print(f"  {role}: {role_text[:50]}...")
            has_pred = True
    if not has_pred:
        print("  (No roles)")

print("\n" + "="*70)
print("Model ready for inference!")
print("="*70)

In [ ]:

# STEP 3: Prepare model and training

# Convert to HuggingFace Dataset
def dict_to_dataset(data_list):
    """Convert list of dicts to HuggingFace Dataset"""
    return Dataset.from_dict({
        'input_ids': [d['input_ids'] for d in data_list],
        'attention_mask': [d['attention_mask'] for d in data_list],
        'token_type_ids': [d['token_type_ids'] for d in data_list],
        'labels': [d['labels'] for d in data_list]
    })

train_hf_dataset = dict_to_dataset(train_dataset)
test_hf_dataset = dict_to_dataset(test_dataset)

print(f"HuggingFace datasets created")
print(f"Train dataset features: {train_hf_dataset.features}")

# Load model
num_labels = len(BIOTagger.LABEL2ID)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

print(f"\nModel loaded: {model_name}")
print(f"Number of labels: {num_labels}")
print(f"Model parameters: {model.num_parameters():,}")

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model moved to: {device}")

# STEP 4: Train model

# Define evaluation metrics
def compute_metrics(p):
    """Compute evaluation metrics"""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (special tokens)
    true_predictions = [
        [BIOTagger.ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [BIOTagger.ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions)
    }

# Training arguments
training_args = TrainingArguments(
    output_dir='./phobert_srl_model',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=10
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf_dataset,
    eval_dataset=test_hf_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training...")
print("="*70)
history = trainer.train()
print("="*70)
print("Training completed!")

# Save model
model.save_pretrained('./phobert_srl_model/final')
tokenizer.save_pretrained('./phobert_srl_model/final')
print("\nModel saved to: ./phobert_srl_model/final")

# STEP 5: Evaluate on test set

print("="*70)
print("EVALUATING MODEL")
print("="*70)

# Evaluate
try:
    results = trainer.evaluate()
    
    print("\nTEST SET RESULTS")
    print("="*70)
    for key, value in results.items():
        if 'loss' in key:
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value:.4f}")
    
except RuntimeError as e:
    print(f"Note: Training not yet complete. Skipping trainer evaluation.")
    print("Run this cell AFTER training completes.")
    results = {}

# Manual evaluation
print("\nManual Prediction Evaluation:")
print("="*70)

predictions = trainer.predict(test_hf_dataset)
preds = np.argmax(predictions.predictions, axis=2)

true_predictions = []
true_labels_list = []

for pred, label in zip(preds, predictions.label_ids):
    true_pred = [BIOTagger.ID2LABEL[p] for (p, l) in zip(pred, label) if l != -100]
    true_label = [BIOTagger.ID2LABEL[l] for (p, l) in zip(pred, label) if l != -100]
    true_predictions.append(true_pred)
    true_labels_list.append(true_label)

print("\nDETAILED CLASSIFICATION REPORT")
print("="*70)
report = classification_report(true_labels_list, true_predictions)
print(report)

# Save results
with open('phobert_srl_results.txt', 'w', encoding='utf-8') as f:
    f.write("TEST SET RESULTS\n")
    f.write("="*70 + "\n\n")
    if results:
        for key, value in results.items():
            if 'loss' in key:
                f.write(f"{key}: {value:.4f}\n")
            else:
                f.write(f"{key}: {value:.4f}\n")
    f.write("\n" + "="*70 + "\n")
    f.write("DETAILED CLASSIFICATION REPORT\n")
    f.write("="*70 + "\n\n")
    f.write(report)

print("\nResults saved to: phobert_srl_results.txt")

# STEP 6: Prediction function and test

class SRLPredictor:
    """Predict semantic roles for new text"""
    
    def __init__(self, model_path='./phobert_srl_model/final'):
        self.model = AutoModelForTokenClassification.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        self.model.eval()
    
    def predict(self, text: str) -> Dict[str, str]:
        """Predict roles for a clause"""
        encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True
        )
        
        input_ids = torch.tensor([encoding['input_ids']]).to(self.device)
        attention_mask = torch.tensor([encoding['attention_mask']]).to(self.device)
        
        # Predict
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        
        logits = outputs.logits[0]
        predictions = torch.argmax(logits, dim=-1).cpu().numpy()
        
        # Convert predictions to roles
        roles = {role: [] for role in BIOTagger.ROLES}
        current_role = None
        current_text = []
        
        tokens = self.tokenizer.convert_ids_to_tokens(encoding['input_ids'])
        
        for token_idx, (pred, token) in enumerate(zip(predictions, tokens)):
            label = BIOTagger.ID2LABEL[pred]
            
            # Skip special tokens
            if token in {'[CLS]', '[SEP]', '[PAD]', '<s>', '</s>'}:
                continue
            
            # Clean token
            clean_token = token.replace('▁', ' ').strip()
            if not clean_token:
                continue
            
            if label == 'O':
                # Outside any role - save current role if exists
                if current_role and current_text:
                    roles[current_role].append(' '.join(current_text))
                    current_role = None
                    current_text = []
            else:
                # Inside a role (B-X or I-X)
                role = label.split('-')[1]
                
                if label.startswith('B-'):
                    # Beginning of role - save previous role if different
                    if current_role and current_role != role and current_text:
                        roles[current_role].append(' '.join(current_text))
                        current_text = []
                    current_role = role
                    current_text = [clean_token]
                else:  # I-X
                    # Continuation of role
                    if current_role == role:
                        current_text.append(clean_token)
                    else:
                        if current_role and current_text:
                            roles[current_role].append(' '.join(current_text))
                        current_role = role
                        current_text = [clean_token]
        
        # Save last role
        if current_role and current_text:
            roles[current_role].append(' '.join(current_text))
        
        # Consolidate and clean
        result = {}
        for role, texts in roles.items():
            if texts:
                result[role] = ', '.join(texts)
            else:
                result[role] = ''
        
        return result

# Test prediction
print("="*70)
print("TESTING PREDICTION")
print("="*70)

predictor = SRLPredictor()

# Test on a few samples
test_texts = [
    test_samples[0]['text'],
    test_samples[1]['text'] if len(test_samples) > 1 else test_samples[0]['text'],
]

for i, text in enumerate(test_texts, 1):
    print(f"\nSample {i}:")
    print(f"Text: {text[:70]}...")
    
    # Ground truth
    sample = test_samples[i-1]
    print(f"Ground truth roles:")
    has_gt = False
    for role, role_text in sample['roles'].items():
        if role_text.strip():
            print(f"  {role}: {role_text[:50]}...")
            has_gt = True
    if not has_gt:
        print("  (No roles)")
    
    # Predictions
    predicted_roles = predictor.predict(text)
    print(f"Predicted roles:")
    has_pred = False
    for role, role_text in predicted_roles.items():
        if role_text.strip():
            print(f"  {role}: {role_text[:50]}...")
            has_pred = True
    if not has_pred:
        print("  (No roles)")

print("\n" + "="*70)
print("Model ready for inference!")
print("="*70)

In [ ]:
def compute_metrics(p):
    """Compute evaluation metrics"""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (special tokens)
    true_predictions = [
        [BIOTagger.ID2LABEL[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [BIOTagger.ID2LABEL[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions)
    }

# Training arguments
training_args = TrainingArguments(
    output_dir='./phobert_srl_model',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=10
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf_dataset,
    eval_dataset=test_hf_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training...")
print("="*70)
history = trainer.train()
print("="*70)
print("Training completed!")

# Save model
model.save_pretrained('./phobert_srl_model/final')
tokenizer.save_pretrained('./phobert_srl_model/final')
print("\nModel saved to: ./phobert_srl_model/final")


In [ ]:
print("="*70)
print("EVALUATING MODEL")
print("="*70)

# Evaluate
try:
    results = trainer.evaluate()
    
    print("\nTEST SET RESULTS")
    print("="*70)
    for key, value in results.items():
        if 'loss' in key:
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value:.4f}")
    
except RuntimeError as e:
    print(f"Note: Training not yet complete. Skipping trainer evaluation.")
    print("Run this cell AFTER training completes.")
    results = {}

# Manual evaluation
print("\nManual Prediction Evaluation:")
print("="*70)

predictions = trainer.predict(test_hf_dataset)
preds = np.argmax(predictions.predictions, axis=2)

true_predictions = []
true_labels_list = []

for pred, label in zip(preds, predictions.label_ids):
    true_pred = [BIOTagger.ID2LABEL[p] for (p, l) in zip(pred, label) if l != -100]
    true_label = [BIOTagger.ID2LABEL[l] for (p, l) in zip(pred, label) if l != -100]
    true_predictions.append(true_pred)
    true_labels_list.append(true_label)

print("\nDETAILED CLASSIFICATION REPORT")
print("="*70)
report = classification_report(true_labels_list, true_predictions)
print(report)

# Save results
with open('phobert_srl_results.txt', 'w', encoding='utf-8') as f:
    f.write("TEST SET RESULTS\n")
    f.write("="*70 + "\n\n")
    if results:
        for key, value in results.items():
            if 'loss' in key:
                f.write(f"{key}: {value:.4f}\n")
            else:
                f.write(f"{key}: {value:.4f}\n")
    f.write("\n" + "="*70 + "\n")
    f.write("DETAILED CLASSIFICATION REPORT\n")
    f.write("="*70 + "\n\n")
    f.write(report)

print("\nResults saved to: phobert_srl_results.txt")


In [ ]:
class SRLPredictor:
    """Predict semantic roles for new text"""
    
    def __init__(self, model_path='./phobert_srl_model/final'):
        self.model = AutoModelForTokenClassification.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        self.model.eval()
    
    def predict(self, text: str) -> Dict[str, str]:
        """Predict roles for a clause"""
        encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True
        )
        
        input_ids = torch.tensor([encoding['input_ids']]).to(self.device)
        attention_mask = torch.tensor([encoding['attention_mask']]).to(self.device)
        
        # Predict
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        
        logits = outputs.logits[0]
        predictions = torch.argmax(logits, dim=-1).cpu().numpy()
        
        # Convert predictions to roles
        roles = {role: [] for role in BIOTagger.ROLES}
        current_role = None
        current_text = []
        
        tokens = self.tokenizer.convert_ids_to_tokens(encoding['input_ids'])
        
        for token_idx, (pred, token) in enumerate(zip(predictions, tokens)):
            label = BIOTagger.ID2LABEL[pred]
            
            # Skip special tokens
            if token in {'[CLS]', '[SEP]', '[PAD]', '<s>', '</s>'}:
                continue
            
            # Clean token
            clean_token = token.replace('▁', ' ').strip()
            if not clean_token:
                continue
            
            if label == 'O':
                # Outside any role - save current role if exists
                if current_role and current_text:
                    roles[current_role].append(' '.join(current_text))
                    current_role = None
                    current_text = []
            else:
                # Inside a role (B-X or I-X)
                role = label.split('-')[1]
                
                if label.startswith('B-'):
                    # Beginning of role - save previous role if different
                    if current_role and current_role != role and current_text:
                        roles[current_role].append(' '.join(current_text))
                        current_text = []
                    current_role = role
                    current_text = [clean_token]
                else:  # I-X
                    # Continuation of role
                    if current_role == role:
                        current_text.append(clean_token)
                    else:
                        if current_role and current_text:
                            roles[current_role].append(' '.join(current_text))
                        current_role = role
                        current_text = [clean_token]
        
        # Save last role
        if current_role and current_text:
            roles[current_role].append(' '.join(current_text))
        
        # Consolidate and clean
        result = {}
        for role, texts in roles.items():
            if texts:
                result[role] = ', '.join(texts)
            else:
                result[role] = ''
        
        return result

# Test prediction
print("="*70)
print("TESTING PREDICTION")
print("="*70)

predictor = SRLPredictor()

# Test on a few samples
test_texts = [
    test_samples[0]['text'],
    test_samples[1]['text'] if len(test_samples) > 1 else test_samples[0]['text'],
]

for i, text in enumerate(test_texts, 1):
    print(f"\nSample {i}:")
    print(f"Text: {text[:70]}...")
    
    # Ground truth
    sample = test_samples[i-1]
    print(f"Ground truth roles:")
    has_gt = False
    for role, role_text in sample['roles'].items():
        if role_text.strip():
            print(f"  {role}: {role_text[:50]}...")
            has_gt = True
    if not has_gt:
        print("  (No roles)")
    
    # Predictions
    predicted_roles = predictor.predict(text)
    print(f"Predicted roles:")
    has_pred = False
    for role, role_text in predicted_roles.items():
        if role_text.strip():
            print(f"  {role}: {role_text[:50]}...")
            has_pred = True
    if not has_pred:
        print("  (No roles)")

print("\n" + "="*70)
print("Model ready for inference!")
print("="*70)